# Reporte Tecnico: API de Analisis de Sentimiento (Modelo Hibrido G68)

**Equipo:** G68 - Hospitality Intelligence
**Fecha:** Enero 2026

## 1. Introduccion y Propuesta de Valor
El objetivo es proveer al sector hotelero una herramienta que no solo clasifique el sentimiento, sino que identifique disparadores criticos de abandono de clientes. Priorizamos el Recall Negativo para asegurar que ninguna queja pase desapercibida.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import sys
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix

# Asegurar que encuentre la carpeta src para el motor hibrido
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from engine.sentiment_engine import analizar_sentimiento_hibrido

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Carga de Datos
data_path = "../data/raw/Big_AHR.csv" if os.path.exists("../data/raw/Big_AHR.csv") else "Big_AHR.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df.dropna(subset=['review_text'], inplace=True)
    df.drop_duplicates(subset=['review_text'], inplace=True)
    print(f"Datos cargados: {len(df)} registros.")
else:
    print("Utilizando datos de prueba.")
    df = pd.DataFrame({
        'rating': [5]*100 + [1]*20 + [3]*20,
        'review_text': ['Excelente estancia']*100 + ['Sucio y con moho']*20 + ['Normalito']*20
    })


## 2. Entrenamiento y Calibracion G68
Entrenamos el modelo base de Machine Learning.

In [ ]:
df['sentiment'] = df['rating'].apply(lambda r: 'Negativo' if r<=2 else ('Neutro' if r==3 else 'Positivo'))

# Balanceo
df_maj = df[df.sentiment=='Positivo']
df_min_neg = resample(df[df.sentiment=='Negativo'], replace=True, n_samples=len(df_maj), random_state=42)
df_min_neu = resample(df[df.sentiment=='Neutro'], replace=True, n_samples=len(df_maj), random_state=42)
df_bal = pd.concat([df_maj, df_min_neg, df_min_neu])

X_train, X_test, y_train, y_test = train_test_split(df_bal['review_text'], df_bal['sentiment'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = CalibratedClassifierCV(LinearSVC(class_weight='balanced'), method='sigmoid')
model.fit(X_train_vec, y_train)
print("Modelo G68 listo.")

## 3. Playground Interactivo
Prueba de sentimientos hibridos.

In [ ]:
test_review = "El hotel es una maravilla, tiene cucarachas en la cama"
prevision, prob, meta = analizar_sentimiento_hibrido(test_review, model, vectorizer)

print(f"RESEÑA: '{test_review}'")
print("-" * 50)
print(f"ML PURO: {meta['ml_original']}")
print(f"AJUSTE G68: {meta['ajuste_semantico']}")
print(f"RESULTADO: {prevision} ({prob})")